Mục tiêu: Xây dựng mô hình dự đoán giá nhà từ bộ dữ liệu kc_house_data_NaN.csv. Thông qua bài toán này, thực hành và minh họa rõ nét 3 trạng thái của một mô hình học máy: Chưa khớp (Underfitting), Quá khớp (Overfitting), và Cân bằng (Balance/Good Fit) bằng cách thay đổi độ phức tạp của đặc trưng và áp dụng kỹ thuật điều chuẩn (Regularization). Tiêu chí đánh giá là hệ số xác định $R^2$.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score

# 1. Đọc dữ liệu
df = pd.read_csv('kc_house_data_NaN.csv')
display(df.head())
# 2. LÀM SẠCH KỸ: Bỏ 'id', 'date' và các cột tọa độ nhạy cảm dễ gây overfit tuyệt đối
# Chỉ giữ lại các cột thông số vật lý của căn nhà (khoảng 16 cột số)
X = df.drop(columns=['id', 'date', 'price'], errors='ignore')
Y = df['price']

# Điền giá trị khuyết nếu có
X = X.fillna(X.mean())

# Chia tập dữ liệu
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# --- TRẠNG THÁI 1: UNDERFIT (Linear Regression thuần túy) ---
scaler_lin = StandardScaler()
X_train_scaled = scaler_lin.fit_transform(X_train)
X_test_scaled = scaler_lin.transform(X_test)

model_lin = LinearRegression()
model_lin.fit(X_train_scaled, Y_train)
print(f"1. Linear Regression (Underfit): {r2_score(Y_test, model_lin.predict(X_test_scaled)):.4f}")

# --- TRẠNG THÁI 2: OVERFIT (PolynomialFeatures bậc 2 hoặc 3) ---
# Dùng degree=2 với ~16 cột sẽ sinh ra khoảng 150 cột tương tác chéo (đủ để thấy sụt giảm nếu không chuẩn hóa khắt khe, hoặc dùng degree=3)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler_poly = StandardScaler()
X_train_poly_scaled = scaler_poly.fit_transform(X_train_poly)
X_test_poly_scaled = scaler_poly.transform(X_test_poly)

model_poly = LinearRegression()
model_poly.fit(X_train_poly_scaled, Y_train)
print(f"2. Polynomial + Linear (Overfit): {r2_score(Y_test, model_poly.predict(X_test_poly_scaled)):.4f}")

# --- TRẠNG THÁI 3: BALANCE (RidgeCV kiểm soát ma trận đa thức) ---
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 5000.0, 10000.0]
model_ridge = RidgeCV(alphas=alphas, cv=5)
model_ridge.fit(X_train_poly_scaled, Y_train)
print(f"3. Polynomial + RidgeCV (Balance): {r2_score(Y_test, model_ridge.predict(X_test_poly_scaled)):.4f}")

,Unnamed: 0,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,0,7129300520,20141013T000000,221900.0,3.0,1.00,1180,5650,1.0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,1,6414100192,20141209T000000,538000.0,3.0,2.25,2570,7242,2.0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,2,5631500400,20150225T000000,180000.0,2.0,1.00,770,10000,1.0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,3,2487200875,20141209T000000,604000.0,4.0,3.00,1960,5000,1.0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,4,1954400510,20150218T000000,510000.0,3.0,2.00,1680,8080,1.0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


1. Linear Regression (Underfit): 0.7009
2. Polynomial + Linear (Overfit): -1.2488
3. Polynomial + RidgeCV (Balance): 0.7555
